In [0]:
import os
import shutil
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
pd.set_option('display.max_columns', None)

In [0]:
SCHEMA_BEST_PRICES = pa.schema([
    ("label", pa.string()),
    ("airline", pa.string()),
    ("flight_number", pa.string()),
    ("stops", pa.int64()),
    ("duration_minutes", pa.int64()),
    ("total_price", pa.float64()),
    ("currency", pa.string()),
])

SCHEMA_ROUND_TRIP = pa.schema([
    ("label", pa.string()),
    ("airline", pa.string()),
    ("flight_number", pa.string()),
    ("stops", pa.int64()),
    ("duration_minutes", pa.int64()),
    ("total_price", pa.float64()),
    ("currency", pa.string()),
    ("outbound_date", pa.string()),
    ("return_date", pa.string()),
])

In [0]:
def read_silver(table):
    # Read a Silver table and return as pandas DataFrame
    import pandas as pd
    
    try:
        df = spark.read.table(table)
        pdf = df.toPandas()
        print(f"[gold] {table} loaded: {len(pdf)} rows")
        return pdf
        
    except Exception as e:
        print(f"[gold] Silver table not found — run Airflight-Silver first. Error: {e}")
        return pd.DataFrame()

In [0]:
import pandas as pd

# Read both Silver tables
direct = read_silver("workspace.silver.airflights_direct")
connecting = read_silver("workspace.silver.airflights_connecting")

# Combine into one DataFrame
combined = pd.concat([direct, connecting], ignore_index=True)
print(f"Combined table: {len(combined)} total rows")

# Check data
if not combined.empty:
    print(f"\nTrip type breakdown:")
    print(combined["trip_type"].value_counts())
    display(combined.head())
    
    # Find best round-trip combinations
    result = round_trip_combos(combined)
    
    if not result.empty:
        print(f"\nFound {len(result)} round-trip combinations")
        display(result)
        
        # Create temp view for materialized view
        result_spark = spark.createDataFrame(result)
        result_spark.createOrReplaceTempView("temp_roundtrips")
    else:
        print("No round-trip combinations found")
else:
    print("No data in Silver tables")

In [0]:
def round_trip_combos(combined):
    # Find cheapest same-airline round-trip by merging outbound + return on matching airport pair
    import pandas as pd
    
    try:
        # Filter outbound flights
        out = combined[combined["trip_type"] == "outbound"].copy()
        # Filter return flights
        ret = combined[combined["trip_type"] == "return"].copy()

        # Merge outbound and return flights on airline and swapped origin/destination
        merged = out.merge(
            ret,
            left_on=["airline", "origin", "destination"],
            right_on=["airline", "destination", "origin"],
            suffixes=("_out", "_ret")
        )

        # Calculate total round-trip price
        merged["total_price"] = merged["price_out"] + merged["price_ret"]

        # Find index of cheapest round-trip for each airline and route
        idx = merged.groupby(["airline", "origin_out", "destination_out"])["total_price"].idxmin()
        best = merged.loc[idx].copy()

        # Build result DataFrame with required columns for round-trip combos
        result = pd.DataFrame({
            "label": best["origin_out"] + " → " + best["destination_out"] + " | " + best["origin_ret"] + " → " + best["destination_ret"],
            "airline": best["airline"],
            "flight_number": best["flight_number_out"] + " + " + best["flight_number_ret"],
            "stops": best["stops_out"] + best["stops_ret"],
            "duration_minutes": best["duration_minutes_out"] + best["duration_minutes_ret"],
            "total_price": best["total_price"],
            "currency": best["currency_out"],
            "outbound_date": best["departure_time_out"],
            "return_date": best["departure_time_ret"],
        })

        # Return DataFrame of best round-trip combos
        return result
        
    except Exception as e:
        print(f"Error in round_trip_combos: {e}")
        return pd.DataFrame()
    return pd.DataFrame

In [0]:
%sql
Select* from temp_roundtrips